# HPO Seed-Stability Check -- Coffee Bean Quality Detection
### Berapa banyak angka test macro-F1 tuned bergerak kalau cuma random seed training yang berubah?

**Konteks:** `CBQD - XAI HPO-Tuned.ipynb` menemukan retrain `08_multitask` dengan hyperparameter
IDENTIK ke `CBQD - HPO Optuna.ipynb` menghasilkan test macro-F1 yang jauh berbeda (0,9698 vs
0,9352 -- selisih 3,5pp, lebih rendah dari baseline pre-HPO-nya sendiri 0,9397). Dua angka itu
BUKAN eksperimen repeated-seed yang bersih -- keduanya cuma kebetulan beda posisi RNG stream
(HPO Optuna membakar ~30 trial dulu sebelum retrain final; verifikasi kita mulai retrain
langsung setelah seeding) -- jadi belum jelas berapa besar variance itu murni dari
stokastisitas training vs dari confound RNG-stream itu sendiri.

**Tujuan notebook ini:** eksperimen terkontrol -- SATU split train/val/test yang sama persis
dipertahankan (identik `CBQD - Training.ipynb`/`CBQD - HPO Optuna.ipynb`/`CBQD - XAI
HPO-Tuned.ipynb`), hyperparameter TETAP (best_params dari `metadata/hpo_final_summary.csv`),
cuma random seed training yang di-reset eksplisit dan diganti 5x per model
(`SEEDS = [42, 43, 44, 45, 46]`) -- total 15 retrain (3 model x 5 seed). Karena split &
hyperparameter tetap, satu-satunya sumber variance yang tersisa di angka test macro-F1 adalah
stokastisitas training itu sendiri (init head classifier, urutan augmentasi, urutan batch,
titik early-stopping) -- bukan pembagian data mana yang kebetulan jadi test.

**Bukan diganti:** ini bukan pengganti repeated k-fold CV (yang mengubah pembagian data,
bukan cuma seed) -- ini diagnostik yang lebih murah untuk menjawab dulu apakah instabilitas
`08_multitask` murni soal training noise, sebelum berinvestasi di CV yang jauh lebih mahal.
Checkpoint tiap seed TIDAK disimpan permanen (murni untuk mengukur variance, bukan mencari
checkpoint produksi) -- hanya metrik yang direkam.


## Section 1 -- Environment & Data Provenance Setup

In [ ]:
# Sub-Step 1.1
# Tujuan: Install dependency tambahan (tanpa menyentuh torch/torchvision)
# Catatan: tidak butuh captum/lightgbm/shap/timm -- notebook ini murni retrain + eval,
# tanpa XAI dan tanpa Model 01/06/07/10.

import os, json
from pathlib import Path

GIT_REPO_URL = "https://github.com/Ardiyanto24/coffee-bean-quality-detection.git"
PROJECT_DIR = "/kaggle/working/coffee-bean-quality-detection"
if not os.path.exists(PROJECT_DIR):
    os.system(f"git clone {GIT_REPO_URL} {PROJECT_DIR}")
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())


In [ ]:
# Sub-Step 1.2
# Tujuan: Konfigurasi kredensial R2 (private dataset jika ada, fallback ke Kaggle Secrets)

_matches = list(Path("/kaggle/input").rglob("r2_credentials.json")) if os.path.exists("/kaggle/input") else []
cred_path = _matches[0] if _matches else None

if cred_path is not None:
    creds = json.loads(cred_path.read_text())
    os.environ["AWS_ACCESS_KEY_ID"] = creds["R2_ACCESS_KEY_ID"]
    os.environ["AWS_SECRET_ACCESS_KEY"] = creds["R2_SECRET_ACCESS_KEY"]
    print("Kredensial R2 dimuat dari private Kaggle Dataset (nilai tidak di-print).")
else:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("R2_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("R2_SECRET_ACCESS_KEY")
    print("Kredensial R2 dimuat dari Kaggle Secrets (nilai tidak di-print).")


In [ ]:
# Sub-Step 1.3
# Tujuan: Tarik dataset + manifest dari R2 (dvc pull) -- checkpoint model TIDAK dibutuhkan
# notebook ini (semua retrain dari awal, tidak ada get_or_train cache-if-exists).

!pip install -q "dvc[s3]"
!dvc pull -v
print("dataset/ ada:", Path("dataset").exists())
print("dataset_preprocessed/ ada:", Path("dataset_preprocessed").exists())
print("manifest_preprocessed.csv ada:", Path("metadata/manifest_preprocessed.csv").exists())
print("hpo_final_summary.csv ada:", Path("metadata/hpo_final_summary.csv").exists())


## Section 2 -- Konfigurasi

In [ ]:
# Sub-Step 2.1
# Tujuan: Flag DRY_RUN + daftar SEEDS untuk repeated-seed sweep

import random
import numpy as np
import torch

DRY_RUN = False  # <-- dry-run (v1) sudah diverifikasi bersih di Kaggle (kernel v1, COMPLETE, tanpa error), full run.

if DRY_RUN:
    SEEDS = [42, 43]
    EPOCHS_PHASE1 = 1
    EPOCHS_PHASE2 = 4
    EARLY_STOP_PATIENCE = 2
else:
    SEEDS = [42, 43, 44, 45, 46]
    EPOCHS_PHASE1 = 5
    EPOCHS_PHASE2 = 45      # sama seperti retrain final CBQD - HPO Optuna.ipynb
    EARLY_STOP_PATIENCE = 10

IMG_SIZE = 224
CLASS_NAMES = ["defect", "longberry", "peaberry", "premium"]
LABEL_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
TYPE_TO_FLAT = {0: LABEL_TO_IDX["premium"], 1: LABEL_TO_IDX["peaberry"], 2: LABEL_TO_IDX["longberry"]}


def set_all_seeds(seed):
    """Dipanggil ULANG tepat sebelum tiap retrain -- bukan cuma sekali di awal notebook.
    Ini kunci eksperimen ini: kalau cuma di-set sekali di awal (seperti notebook lain di
    project ini), posisi RNG stream saat model ke-2/ke-3/dst mulai dilatih sudah bergeser
    jauh dari titik seed semula -- confound yang sama persis yang membuat perbandingan
    CBQD - HPO Optuna.ipynb vs CBQD - XAI HPO-Tuned.ipynb tidak bisa dipercaya sebagai
    repeated-seed yang bersih (lihat markdown di Sub-Step 0)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_all_seeds(SEEDS[0])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DRY_RUN={DRY_RUN} | device={device} | SEEDS={SEEDS} | epochs phase1/phase2={EPOCHS_PHASE1}/{EPOCHS_PHASE2}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("GPU:", gpu_name)
    if "P100" in gpu_name:
        raise RuntimeError(
            f"GPU allocated is {gpu_name}, incompatible with the preinstalled PyTorch "
            "build (no Pascal/sm_60 kernels). Re-push the kernel with "
            "kernel-metadata.json machine_shape=NvidiaTeslaT4 to force a T4 allocation."
        )


In [ ]:
# Sub-Step 2.2
# Tujuan: Muat best hyperparameter hasil HPO (metadata/hpo_final_summary.csv) per model --
# TETAP untuk seluruh seed sweep, cuma seed training yang berubah.

import pandas as pd

hpo_summary = pd.read_csv("metadata/hpo_final_summary.csv").set_index("model")


def get_best_params(model_name, keys):
    row = hpo_summary.loc[model_name]
    params = {}
    for k in keys:
        v = row[f"param_{k}"]
        params[k] = int(v) if k in ("batch_size", "scheduler_patience") else float(v)
    return params


SHARED_KEYS = ["lr_phase1", "lr_phase2", "weight_decay", "batch_size", "scheduler_factor", "scheduler_patience"]
bp_convnext = get_best_params("05_convnext_tiny", SHARED_KEYS)
bp_multitask = get_best_params("08_multitask", SHARED_KEYS + ["type_loss_weight"])
bp_noise_robust = get_best_params("09_noise_robust", SHARED_KEYS + ["label_smoothing", "mislabel_weight", "mistake_threshold"])
for _name, _bp in [("05_convnext_tiny", bp_convnext), ("08_multitask", bp_multitask), ("09_noise_robust", bp_noise_robust)]:
    print(f"[{_name}] best params (tetap sepanjang seed sweep): {_bp}")


## Section 3 -- Data: Manifest, Dataset, Transform, DataLoader (SATU split, tetap)

In [ ]:
# Sub-Step 3.1
# Tujuan: Load manifest, definisikan fit/val/test DataFrame -- split identik ke 3 notebook
# sebelumnya, TIDAK berubah sepanjang seed sweep.

manifest = pd.read_csv("metadata/manifest_preprocessed.csv")
PREP_DIR = Path("dataset_preprocessed")
RAW_DIR = Path("dataset")

train_pool = manifest[manifest["split"] == "train"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)

fit_df = train_pool[train_pool["cv_fold"].isin([1, 2, 3])].reset_index(drop=True)
val_df = train_pool[train_pool["cv_fold"] == 0].reset_index(drop=True)

print(f"fit={len(fit_df)}  val={len(val_df)}  test={len(test_df)}")


In [ ]:
# Sub-Step 3.2
# Tujuan: Dataset & transform; DataLoader test TETAP (dipakai evaluasi semua seed & model)
# + make_loaders() untuk fit/val yang dibangun ULANG tiap seed (supaya shuffle order-nya
# ikut ke-reset oleh seed baru, bukan cuma bobot modelnya).

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
EVAL_BATCH_SIZE = 32

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=180),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class BeanDataset(Dataset):
    def __init__(self, df, root_dir, transform, weights=None):
        self.paths = [root_dir / p for p in df["image_path"]]
        self.labels = [LABEL_TO_IDX[l] for l in df["label"]]
        self.transform = transform
        self.weights = weights if weights is not None else [1.0] * len(df)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.labels[idx], self.weights[idx]


class MultiTaskDataset(Dataset):
    TYPE_MAP = {"premium": 0, "peaberry": 1, "longberry": 2}

    def __init__(self, df, root_dir, transform):
        self.paths = [root_dir / p for p in df["image_path"]]
        self.damage_labels = [1 if l == "defect" else 0 for l in df["label"]]
        self.type_labels = [self.TYPE_MAP.get(l, -1) for l in df["label"]]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.damage_labels[idx], self.type_labels[idx]


def make_loaders(dataset_cls, fit_kwargs, val_kwargs, batch_size):
    fit_loader = DataLoader(dataset_cls(fit_df, PREP_DIR, train_transform, **fit_kwargs),
                             batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(dataset_cls(val_df, PREP_DIR, eval_transform, **val_kwargs),
                             batch_size=batch_size, shuffle=False, num_workers=2)
    return fit_loader, val_loader


test_loader = DataLoader(BeanDataset(test_df, PREP_DIR, eval_transform),
                          batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2)
print("DataLoaders siap. test_loader ini akan dipakai SAMA PERSIS untuk evaluasi tiap seed.")


## Section 4 -- Fungsi Utilitas Model (identik `CBQD - HPO Optuna.ipynb`)

In [ ]:
# Sub-Step 4.1
# Tujuan: evaluate() & evaluate_combined()

import torch.nn as nn
from sklearn.metrics import f1_score, accuracy_score, classification_report

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
    return {"macro_f1": macro_f1, "accuracy": acc, "report": report}


@torch.no_grad()
def evaluate_combined(predict_fn, loader, device):
    all_preds, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(device)
        preds = predict_fn(images)
        all_preds.extend(list(preds))
        all_labels.extend(labels.tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
    return {"macro_f1": macro_f1, "accuracy": acc, "report": report}


In [ ]:
# Sub-Step 4.2
# Tujuan: set_backbone_frozen() + train_one_model_hpo()/train_multitask_hpo() -- hyperparameter
# sebagai ARGUMEN (best_params tetap), dipanggil ulang tiap seed dengan model & loader baru.

import copy


def set_backbone_frozen(model, head_module, frozen: bool):
    head_param_ids = set(id(p) for p in head_module.parameters())
    for p in model.parameters():
        p.requires_grad = (id(p) in head_param_ids) or (not frozen)


def _train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    for images, labels, weights in loader:
        images, labels, weights = images.to(device), labels.to(device), weights.to(device).float()
        optimizer.zero_grad()
        outputs = model(images)
        per_sample_loss = criterion(outputs, labels)
        loss = (per_sample_loss * weights).mean() if per_sample_loss.dim() > 0 else per_sample_loss
        loss.backward()
        optimizer.step()


def train_one_model_hpo(model, head_module, fit_loader, val_loader, device, model_name,
                         lr_phase1, lr_phase2, weight_decay, scheduler_factor, scheduler_patience,
                         epochs_phase2, criterion=None):
    model = model.to(device)
    criterion = criterion if criterion is not None else nn.CrossEntropyLoss(reduction="none")
    best_state, best_val_f1, patience_counter = None, -1.0, 0

    set_backbone_frozen(model, head_module, frozen=True)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr_phase1, weight_decay=weight_decay
    )
    for epoch in range(EPOCHS_PHASE1):
        _train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device)
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1

    set_backbone_frozen(model, head_module, frozen=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_phase2, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=scheduler_factor, patience=scheduler_patience)
    patience_counter = 0
    for epoch in range(epochs_phase2):
        _train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device)
        scheduler.step(val_metrics["macro_f1"])
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_f1


@torch.no_grad()
def _mt_val_f1(model, loader, device):
    model.eval()
    preds, labels_flat = [], []
    for images, damage_labels, type_labels in loader:
        images = images.to(device)
        out_damage, out_type = model(images)
        damage_pred = out_damage.argmax(dim=1).cpu().numpy()
        type_pred = out_type.argmax(dim=1).cpu().numpy()
        combined = np.where(damage_pred == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT[t] for t in type_pred])
        preds.extend(list(combined))
        dmg_np, typ_np = damage_labels.numpy(), type_labels.numpy()
        true_flat = np.where(dmg_np == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT.get(t, -1) for t in typ_np])
        labels_flat.extend(true_flat.tolist())
    return f1_score(labels_flat, preds, average="macro", zero_division=0)


def train_multitask_hpo(model, fit_loader, val_loader, device, model_name,
                         lr_phase1, lr_phase2, weight_decay, scheduler_factor, scheduler_patience,
                         epochs_phase2, type_loss_weight=1.0):
    model = model.to(device)
    ce_damage, ce_type = nn.CrossEntropyLoss(), nn.CrossEntropyLoss()
    best_state, best_val_f1, patience_counter = None, -1.0, 0

    def run_epoch(optimizer):
        model.train()
        for images, damage_labels, type_labels in fit_loader:
            images = images.to(device); damage_labels = damage_labels.to(device); type_labels = type_labels.to(device)
            optimizer.zero_grad()
            out_damage, out_type = model(images)
            loss = ce_damage(out_damage, damage_labels)
            mask = type_labels >= 0
            if mask.any():
                loss = loss + type_loss_weight * ce_type(out_type[mask], type_labels[mask])
            loss.backward()
            optimizer.step()

    for p in model.backbone.parameters():
        p.requires_grad = False
    optimizer = torch.optim.AdamW(
        list(model.head_damage.parameters()) + list(model.head_type.parameters()),
        lr=lr_phase1, weight_decay=weight_decay,
    )
    for epoch in range(EPOCHS_PHASE1):
        run_epoch(optimizer)
        val_f1 = _mt_val_f1(model, val_loader, device)
        if val_f1 > best_val_f1:
            best_val_f1, best_state, patience_counter = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1

    for p in model.parameters():
        p.requires_grad = True
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_phase2, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=scheduler_factor, patience=scheduler_patience)
    patience_counter = 0
    for epoch in range(epochs_phase2):
        run_epoch(optimizer)
        val_f1 = _mt_val_f1(model, val_loader, device)
        scheduler.step(val_f1)
        if val_f1 > best_val_f1:
            best_val_f1, best_state, patience_counter = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_f1


In [ ]:
# Sub-Step 4.3
# Tujuan: build_model() (convnext_tiny & efficientnet_b0) + EfficientNetMultiTask

from torchvision import models as tv_models


def build_model(arch, num_classes):
    if arch == "efficientnet_b0":
        m = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    elif arch == "convnext_tiny":
        m = tv_models.convnext_tiny(weights=tv_models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    else:
        raise ValueError(f"Arsitektur tidak dikenal: {arch}")
    return m, head


class EfficientNetMultiTask(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = backbone.classifier[-1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.head_damage = nn.Linear(in_f, 2)
        self.head_type = nn.Linear(in_f, 3)

    def forward(self, x):
        feats = self.backbone(x)
        return self.head_damage(feats), self.head_type(feats)


In [ ]:
# Sub-Step 4.4
# Tujuan: handcrafted_features()/build_feature_matrix() -- dibutuhkan utk mistake_score
# 09_noise_robust (dihitung SEKALI, tidak ikut seed sweep -- lihat catatan metodologis di
# Sub-Step 5.1: mistake_score bukan sumber randomness training, cuma proxy label statis).

import cv2


def handcrafted_features(path):
    bgr = cv2.imread(str(path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    gray_f = gray.astype(np.float32)
    thresh = gray_f.mean() - 0.6 * gray_f.std()
    mask = gray_f < thresh
    h, w = gray.shape
    if mask.sum() > 0:
        ys, xs = np.where(mask)
        area_frac = mask.sum() / (h * w)
        bbox_h, bbox_w = float(ys.max() - ys.min()), float(xs.max() - xs.min())
        bbox_ratio = max(bbox_h, bbox_w) / max(min(bbox_h, bbox_w), 1e-6)
        cy, cx = ys.mean(), xs.mean()
        center_offset = float(np.hypot(cy - h / 2, cx - w / 2) / (h / 2))
    else:
        area_frac = bbox_ratio = center_offset = np.nan

    edges = cv2.Canny(gray, 100, 200)
    feats = {
        "mean_r": rgb[:, :, 0].mean(), "mean_g": rgb[:, :, 1].mean(), "mean_b": rgb[:, :, 2].mean(),
        "std_r": rgb[:, :, 0].std(), "std_g": rgb[:, :, 1].std(), "std_b": rgb[:, :, 2].std(),
        "edge_density": edges.mean() / 255, "variance": float(np.var(gray)),
        "area_frac": area_frac, "bbox_ratio": bbox_ratio, "center_offset": center_offset,
    }
    return feats


FEATURE_COLS = ["mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b",
                "edge_density", "variance", "area_frac", "bbox_ratio", "center_offset"]


def build_feature_matrix(df):
    rows = [handcrafted_features(RAW_DIR / p) for p in df["orig_path"]]
    X = pd.DataFrame(rows)[FEATURE_COLS].values
    y = np.array([LABEL_TO_IDX[l] for l in df["label"]])
    return X, y


## Section 5 -- Seed Sweep: 5 Retrain Independen per Model, Split &amp; Hyperparameter Tetap

Tiap model diretrain `len(SEEDS)` kali. `set_all_seeds(seed)` dipanggil ULANG tepat sebelum
loader &amp; model dibangun di tiap iterasi -- ini yang membuat tiap retrain jadi independen
secara statistik (init head, urutan augmentasi, urutan batch semua ikut berubah), bukan cuma
`torch.manual_seed()` sekali di awal notebook. Checkpoint TIDAK disimpan -- hanya metrik test
yang direkam ke `results`.

In [ ]:
# Sub-Step 5.1
# Tujuan: Precompute mistake_score untuk 09_noise_robust SEKALI (identik CBQD - HPO Optuna.ipynb)
# -- ini proxy label statis dari RandomForest OOF, BUKAN bagian dari stokastisitas training
# yang sedang diukur notebook ini, jadi sengaja tidak ikut di-reset tiap seed.

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier

X_fit_m9, y_fit_m9 = build_feature_matrix(fit_df)
_sgkf_noise = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
_rf_noise = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
_proba_oof = cross_val_predict(_rf_noise, X_fit_m9, y_fit_m9, cv=_sgkf_noise,
                                groups=fit_df["cluster_id"].values, method="predict_proba")
_true_proba = _proba_oof[np.arange(len(y_fit_m9)), y_fit_m9]
_max_proba = _proba_oof.max(axis=1)
MISTAKE_SCORE = _max_proba - _true_proba

flagged_mask = MISTAKE_SCORE > bp_noise_robust["mistake_threshold"]
sample_weights_fit_m9 = np.where(flagged_mask, bp_noise_robust["mislabel_weight"], 1.0)
print(f"[09_noise_robust] kandidat mislabel (threshold={bp_noise_robust['mistake_threshold']:.4f}): "
      f"{flagged_mask.sum()} / {len(fit_df)} -- TETAP sepanjang seed sweep")

results = []


In [ ]:
# Sub-Step 5.2
# Tujuan: Seed sweep -- 05_convnext_tiny

for seed in SEEDS:
    set_all_seeds(seed)
    fit_loader, val_loader = make_loaders(BeanDataset, {}, {}, bp_convnext["batch_size"])
    model, head = build_model("convnext_tiny", num_classes=4)
    model, val_f1 = train_one_model_hpo(
        model, head, fit_loader, val_loader, device, f"05_convnext_tiny_seed{seed}",
        lr_phase1=bp_convnext["lr_phase1"], lr_phase2=bp_convnext["lr_phase2"],
        weight_decay=bp_convnext["weight_decay"], scheduler_factor=bp_convnext["scheduler_factor"],
        scheduler_patience=bp_convnext["scheduler_patience"], epochs_phase2=EPOCHS_PHASE2,
    )
    test_metrics = evaluate(model, test_loader, device)
    row = {"model": "05_convnext_tiny", "seed": seed, "val_macro_f1": val_f1,
           "test_macro_f1": test_metrics["macro_f1"], "test_accuracy": test_metrics["accuracy"]}
    for cls in CLASS_NAMES:
        row[f"test_recall_{cls}"] = test_metrics["report"][cls]["recall"]
    results.append(row)
    print(f"[05_convnext_tiny seed={seed}] val_f1={val_f1:.4f}  test_f1={test_metrics['macro_f1']:.4f}")
    del model
    torch.cuda.empty_cache()


In [ ]:
# Sub-Step 5.3
# Tujuan: Seed sweep -- 09_noise_robust (mistake_score & threshold TETAP, cuma seed training berubah)

for seed in SEEDS:
    set_all_seeds(seed)
    fit_loader, val_loader = make_loaders(BeanDataset, {"weights": sample_weights_fit_m9.tolist()}, {}, bp_noise_robust["batch_size"])
    criterion = nn.CrossEntropyLoss(label_smoothing=bp_noise_robust["label_smoothing"], reduction="none")
    model, head = build_model("efficientnet_b0", num_classes=4)
    model, val_f1 = train_one_model_hpo(
        model, head, fit_loader, val_loader, device, f"09_noise_robust_seed{seed}",
        lr_phase1=bp_noise_robust["lr_phase1"], lr_phase2=bp_noise_robust["lr_phase2"],
        weight_decay=bp_noise_robust["weight_decay"], scheduler_factor=bp_noise_robust["scheduler_factor"],
        scheduler_patience=bp_noise_robust["scheduler_patience"], epochs_phase2=EPOCHS_PHASE2, criterion=criterion,
    )
    test_metrics = evaluate(model, test_loader, device)
    row = {"model": "09_noise_robust", "seed": seed, "val_macro_f1": val_f1,
           "test_macro_f1": test_metrics["macro_f1"], "test_accuracy": test_metrics["accuracy"]}
    for cls in CLASS_NAMES:
        row[f"test_recall_{cls}"] = test_metrics["report"][cls]["recall"]
    results.append(row)
    print(f"[09_noise_robust seed={seed}] val_f1={val_f1:.4f}  test_f1={test_metrics['macro_f1']:.4f}")
    del model
    torch.cuda.empty_cache()


In [ ]:
# Sub-Step 5.4
# Tujuan: Seed sweep -- 08_multitask

for seed in SEEDS:
    set_all_seeds(seed)
    fit_loader, val_loader = make_loaders(MultiTaskDataset, {}, {}, bp_multitask["batch_size"])
    model = EfficientNetMultiTask()
    model, val_f1 = train_multitask_hpo(
        model, fit_loader, val_loader, device, f"08_multitask_seed{seed}",
        lr_phase1=bp_multitask["lr_phase1"], lr_phase2=bp_multitask["lr_phase2"],
        weight_decay=bp_multitask["weight_decay"], scheduler_factor=bp_multitask["scheduler_factor"],
        scheduler_patience=bp_multitask["scheduler_patience"], epochs_phase2=EPOCHS_PHASE2,
        type_loss_weight=bp_multitask["type_loss_weight"],
    )

    def multitask_predict_fn(images, _model=model):
        out_damage, out_type = _model(images)
        damage_pred = out_damage.argmax(dim=1).cpu().numpy()
        type_pred = out_type.argmax(dim=1).cpu().numpy()
        return np.where(damage_pred == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT[t] for t in type_pred])

    test_metrics = evaluate_combined(multitask_predict_fn, test_loader, device)
    row = {"model": "08_multitask", "seed": seed, "val_macro_f1": val_f1,
           "test_macro_f1": test_metrics["macro_f1"], "test_accuracy": test_metrics["accuracy"]}
    for cls in CLASS_NAMES:
        row[f"test_recall_{cls}"] = test_metrics["report"][cls]["recall"]
    results.append(row)
    print(f"[08_multitask seed={seed}] val_f1={val_f1:.4f}  test_f1={test_metrics['macro_f1']:.4f}")
    del model
    torch.cuda.empty_cache()


## Section 6 -- Konsolidasi &amp; Perbandingan

In [ ]:
# Sub-Step 6.1
# Tujuan: Simpan detail per (model, seed) + ringkasan mean/std/min/max per model

Path("metadata").mkdir(exist_ok=True)

results_df = pd.DataFrame(results)
results_df.to_csv("metadata/hpo_seed_stability_details.csv", index=False)

summary_rows = []
for name, group in results_df.groupby("model"):
    summary_rows.append({
        "model": name,
        "n_seeds": len(group),
        "test_macro_f1_mean": group["test_macro_f1"].mean(),
        "test_macro_f1_std": group["test_macro_f1"].std(),
        "test_macro_f1_min": group["test_macro_f1"].min(),
        "test_macro_f1_max": group["test_macro_f1"].max(),
    })
summary_df = pd.DataFrame(summary_rows).sort_values("test_macro_f1_mean", ascending=False).reset_index(drop=True)
summary_df.to_csv("metadata/hpo_seed_stability_summary.csv", index=False)

status_text = "BELUM final (seed/epoch kecil)" if DRY_RUN else "hasil run penuh"
print(f"DRY_RUN={DRY_RUN} -- angka di bawah ini {status_text}")
print()
pd.set_option("display.max_columns", None, "display.width", 200)
print("=== Detail per (model, seed) ===")
print(results_df[["model", "seed", "val_macro_f1", "test_macro_f1"]].round(4).to_string(index=False))
print()
print("=== Ringkasan: mean +/- std test macro-F1 across seeds ===")
print(summary_df.round(4).to_string(index=False))


In [ ]:
# Sub-Step 6.2
# Tujuan: Bandingkan ke referensi yang sudah ada -- CV 4-fold pre-HPO, laporan HPO (1 run),
# dan retrain verifikasi XAI (1 run) -- supaya jelas apakah repeated-seed ini menambah bukti
# baru atau cuma menegaskan yang sudah dicurigai.

cv_4fold = pd.read_csv("metadata/cv_4fold_summary.csv").set_index("model")
hpo_report = hpo_summary["tuned_test_macro_f1"]  # 1 angka per model, dari CBQD - HPO Optuna.ipynb

# Angka retrain verifikasi CBQD - XAI HPO-Tuned.ipynb (hardcode -- sudah final & tercatat di
# commit f3c2fd7, bukan file terpisah yang bisa dibaca ulang di sini).
xai_verify = {"08_multitask": 0.9352, "05_convnext_tiny": 0.9267, "09_noise_robust": 0.9739}

comparison_rows = []
for _, row in summary_df.iterrows():
    name = row["model"]
    comparison_rows.append({
        "model": name,
        "cv4fold_pre_hpo_mean": cv_4fold.loc[name, "test_macro_f1_mean"],
        "cv4fold_pre_hpo_std": cv_4fold.loc[name, "test_macro_f1_std"],
        "hpo_report_1run": float(hpo_report.loc[name]),
        "xai_verify_1run": xai_verify[name],
        "seed_sweep_mean": row["test_macro_f1_mean"],
        "seed_sweep_std": row["test_macro_f1_std"],
        "seed_sweep_n": row["n_seeds"],
    })
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv("metadata/hpo_seed_stability_vs_references.csv", index=False)
print(comparison_df.round(4).to_string(index=False))


In [ ]:
# Sub-Step 6.3
# Tujuan: Interpretasi ringkas -- dibaca manual bersama tabel Sub-Step 6.2

print("""
Cara membaca metadata/hpo_seed_stability_vs_references.csv:
- seed_sweep_std yang BESAR (mendekati atau melebihi cv4fold_pre_hpo_std) berarti sebagian
  besar variance yang kita lihat sebelumnya (antara laporan HPO 1-run vs retrain verifikasi
  XAI) memang murni stokastisitas training -- bukan sesuatu yang aneh, ini karakteristik
  dataset sekecil ini (~700 gambar fit), konsisten dengan std yang sudah terlihat di CV
  4-fold pre-HPO.
- seed_sweep_std yang JAUH LEBIH KECIL dari cv4fold_pre_hpo_std berarti stokastisitas
  training BUKAN sumber utama -- variance yang terlihat sebelumnya lebih mungkin datang dari
  confound RNG-stream (HPO Optuna membakar banyak random draw sebelum retrain final-nya)
  atau dari sensitivitas terhadap fold mana yang jadi test (baru bisa dipastikan lewat
  repeated k-fold CV, bukan notebook ini).
- hpo_report_1run dan xai_verify_1run yang jatuh JAUH DI LUAR rentang
  [seed_sweep_mean - 2*seed_sweep_std, seed_sweep_mean + 2*seed_sweep_std] mengindikasikan
  angka itu outlier bahkan terhadap distribusi seed murni -- alasan kuat untuk tidak
  mempercayainya sebagai representasi performa model yang wajar.
- Kalau seed_sweep_mean SEMUA model (bukan cuma 08_multitask) berdekatan satu sama lain
  dalam rentang overlap std masing-masing, itu tanda peringkat "siapa menang" antar 3 model
  ini TIDAK bisa ditentukan dari macro-F1 saja pada skala dataset ini -- perlu kriteria lain
  (mis. kestabilan/std itu sendiri, biaya inferensi, atau recall per-kelas yang paling
  relevan ke use-case) untuk memutuskan model produksi.

Langkah selanjutnya yang disarankan tergantung hasil di atas: kalau seed_sweep_std sudah
menjelaskan sebagian besar instabilitas yang dicurigai, repeated k-fold CV (lebih mahal)
mungkin tidak lagi prioritas -- cukup laporkan mean+/-std dari seed sweep ini sebagai angka
resmi tiap model. Kalau belum, repeated k-fold CV tetap perlu untuk mengisolasi kontribusi
"pembagian data mana yang jadi test" secara terpisah dari stokastisitas training.
""")
